# Coding Nights 3.0 — Night 1, Part 1
### Big-O Thinking, HashMaps & Sets
*Women in Tech × IEEE NTU Student Branch*

---

Welcome! This notebook goes with tonight's workshop. You can run and modify all of the code.

| Section | Topic |
|---|---|
| 1 | Why complexity matters |
| 2 | Big-O: growth rate, not speed |
| 3 | Common complexity classes |
| 4 | Time vs. space |
| 5 | 🎮 Quiz: what's the complexity? |
| 6 | HashMaps & Sets |
| 7 | Pattern: Frequency Counter |
| 8 | Problem: Contains Duplicate |
| 9 | Pattern: Complement Lookup |
| 10 | Problem: Two Sum |
| 11 | ✏️ Practice problems |

**How to use it**
- New to notebooks? Read **How to run a cell** just below.
- Run the **setup cell** first, then run the cells from top to bottom with **Shift + Enter**.
- Before you run a **trace** cell, try to **predict** what it will print.
- Quiz answers and exercise solutions are hidden in **Reveal** boxes. Try the problem first, then look.

When you finish, continue with **`Night1_Part2_Patterns.ipynb`** (two pointers and sliding windows).

## ▶️ How to run a cell

This notebook is made of **cells**: boxes that hold either text (like this one) or Python code (grey boxes with `[ ]:` on the left).

**To run a code cell:**
1. **Click** inside the cell.
2. Press **Shift + Enter**, or click the **▶ Run** button in the toolbar at the top.
3. Any output appears just **below** the cell, and the cursor moves on to the next cell.

**What the `[ ]` on the left means:**

| You see | It means |
|---|---|
| `[ ]` | Not run yet |
| `[*]` | Running right now, so wait a moment |
| `[3]` | Finished. The number shows the order in which cells were run |

**Tips**
- Run the cells **in order, from top to bottom**. Later cells use code from earlier ones.
- Seeing `NameError: name '...' is not defined`? You probably skipped a cell above. Go back and run it.
- Something stuck or behaving oddly? Use **Kernel → Restart & Run All** (in newer Jupyter: **Run → Restart Kernel and Run All Cells**) to start fresh.
- You can edit any code cell. Just change it and run it again.

👉 **Try it now:** click the setup cell below and press **Shift + Enter**. You should see `✅ Setup complete`.

## What you'll learn tonight

1. **Recognise the patterns** behind common coding problems
2. **Choose the right data structure** for the job
3. **Work out how fast your solution is** (its time complexity)

| Topic | The idea in one line |
|---|---|
| **Big-O thinking** | Read any code and predict how its runtime grows |
| **HashMaps & Sets** | Turn O(n²) into O(n) with O(1) lookups |
| **Two pointers** *(Part 2)* | Move through arrays with two coordinated pointers |
| **Interview method** *(Part 2)* | A 5-step plan for approaching any coding problem |

In [ ]:
# ── Setup: run this cell first ─────────────────────────────────────────
# Small display helpers so every result prints as a clean, scannable table.
# You don't need to understand this cell; just run it.
import math, random, time
from collections import Counter

LINE = 64

def banner(title):
    """Print a boxed section header."""
    print("═" * LINE)
    print(f"  {title}")
    print("═" * LINE)

def short(value, width=36):
    """repr() of a value, trimmed so long lists don't break the table."""
    text = repr(value)
    return text if len(text) <= width else text[: width - 1] + "…"

def show_table(rows, headers):
    """Print a list of rows as an aligned text table."""
    rows = [[str(cell) for cell in row] for row in rows]
    widths = [max([len(h)] + [len(r[i]) for r in rows]) for i, h in enumerate(headers)]
    fmt = lambda cells: " │ ".join(c.ljust(w) for c, w in zip(cells, widths))
    print(fmt(headers))
    print("─┼─".join("─" * w for w in widths))
    for row in rows:
        print(fmt(row))

def check(fn, cases, name=None):
    """Run fn(*args) for each (args, expected) pair and print a ✅/❌ table."""
    rows, passed = [], 0
    for i, (args, expected) in enumerate(cases, 1):
        shown_input = ", ".join(short(a, 30) for a in args)   # captured BEFORE fn can modify it
        try:
            got = fn(*args)
        except Exception as e:
            got = f"{type(e).__name__}: {e}"
        ok = got == expected
        passed += ok
        rows.append([i, shown_input, short(expected), short(got), "✅" if ok else "❌"])
    banner(f"Checking your answer: {name or fn.__name__}")
    show_table(rows, ["#", "input", "correct answer", "your answer", ""])
    print("─" * LINE)
    if all(row[3] == "None" for row in rows):
        print("⏳ Not attempted yet: replace `pass` with your code, run that cell, then re-run this one.")
    elif passed == len(cases):
        print(f"🎉 {passed}/{len(cases)} passed. All correct!")
    else:
        print(f"🔁 {passed}/{len(cases)} passed. Look at the ❌ rows and try again.")

print("✅ Setup complete: banner(), show_table() and check() are ready.")

---
## 1. Why complexity matters

Imagine a list of **n = 100,000** numbers, and you need to find a pair that meets some condition.

| Approach | Idea | Work for n = 100,000 |
|---|---|---|
| **Nested loops: O(n²)** | Check every possible pair | ~10,000,000,000 checks (10 billion) |
| **HashMap / Set: O(n)** | Store what you've already seen | ~100,000 lookups |

That is a **100,000× difference**: the gap between "instant" and "go and make a coffee".

The chain to remember:

> **If input size ↑ and work per item ↑, then runtime ↑**

> 💡 *The goal isn't just to get the right answer. It's to find the pattern that gets you there efficiently.*

Let's **count** the work instead of taking it on trust.

In [ ]:
def count_pair_checks(n):
    """How many comparisons does 'check every pair' make?"""
    checks = 0
    for i in range(n):
        for j in range(i + 1, n):
            checks += 1
    return checks

def count_single_pass(n):
    """How many steps does one pass through the list take?"""
    steps = 0
    for i in range(n):
        steps += 1
    return steps

banner("Pair checks vs. a single pass")
show_table(
    [[f"{n:,}", f"{count_pair_checks(n):,}", f"{count_single_pass(n):,}"] for n in [10, 100, 1_000, 3_000]],
    ["n", "pair checks  O(n²)", "single pass  O(n)"],
)

Notice how the pair count grows much faster: going from n = 1,000 to n = 3,000 (×3) makes the pair count about **×9**.

For n = 100,000, checking every pair is billions of steps, while one pass is just 100,000. Big-O calls the first **O(n²)** and the second **O(n)**. We care about the *shape* of the growth, not the exact number.

In [ ]:
n = 100_000
banner(f"How much work for n = {n:,}?")
show_table([
    ["Ordered pairs", "n × n",      f"{n * n:,}"],
    ["Unique pairs",  "n(n − 1)/2", f"{n * (n - 1) // 2:,}"],
    ["Single pass",   "n",          f"{n:,}"],
], ["approach", "formula", "steps"])

---
## 2. Big-O: think growth rate, not speed

**How does your code scale?**

- ⏱️ **Ignore the stopwatch.** Runtime depends on the hardware, the language and the implementation. Your laptop and a server give different times for the same code.
- 📏 **Focus on the input size.** Ask: *what happens when n gets bigger?*
- 📈 **Look at the growth.** Ask: *how much more work happens when n doubles?*

Big-O describes the **shape** of the curve: how the work grows as n grows. Run the cell below to see the common shapes side by side.

In [ ]:
import math
import matplotlib.pyplot as plt

ns = list(range(1, 11))
curves = {
    "O(1)":       [1 for n in ns],
    "O(log n)":   [math.log2(n) for n in ns],
    "O(n)":       [n for n in ns],
    "O(n log n)": [n * math.log2(n) for n in ns],
    "O(n²)":      [n * n for n in ns],
}

plt.figure(figsize=(8, 5))
for label, ys in curves.items():
    plt.plot(ns, ys, marker="o", label=label)
plt.title("Work vs input size (n)")
plt.xlabel("input size n")
plt.ylabel("work")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### The doubling test

A quick way to recognise a complexity class: **double n and see what happens to the work.**

In [ ]:
growth = [
    ("O(1)",       lambda n: 1),
    ("O(log n)",   lambda n: math.log2(n)),
    ("O(n)",       lambda n: n),
    ("O(n log n)", lambda n: n * math.log2(n)),
    ("O(n²)",      lambda n: n * n),
]

rows = []
for label, work in growth:
    a, b = work(1000), work(2000)
    rows.append([label, f"{a:,.0f}", f"{b:,.0f}", f"×{b / a:.2f}"])

banner("The doubling test: n = 1,000 → n = 2,000")
show_table(rows, ["class", "work at n=1,000", "work at n=2,000", "change"])

| When n doubles, work… | Complexity |
|---|---|
| stays the same | O(1) |
| goes up by a tiny fixed amount (one more step) | O(log n) |
| doubles | O(n) |
| slightly more than doubles | O(n log n) |
| **quadruples** | O(n²) |

---
## 3. Common complexity classes at a glance

From fastest to slowest:

| Big-O | Name | Where you see it | Takeaway |
|---|---|---|---|
| **O(1)** | Constant | HashMap & set lookups | Your target for lookups |
| **O(log n)** | Logarithmic | Binary search: cut the search space in half every step | Reduce the search space |
| **O(n)** | Linear | One pass through an array; two pointers get you here | Your goal for many problems |
| **O(n log n)** | Linearithmic | Merge sort and heap sort, the usual cost of efficient sorting | Common baseline for sorting |
| **O(n²)** | Quadratic | Nested loops, checking every pair | Look for ways to avoid this |

Below is one small example of each. Cover the comment above each function, guess its complexity, then check your guess.

In [ ]:
nums = [3, 8, 1, 9, 4, 7, 2]

# O(1): one step, no matter how long the list is
def get_first(nums):
    return nums[0]

# O(log n): binary search on a SORTED list halves the search space every step
def binary_search(sorted_nums, target):
    lo, hi = 0, len(sorted_nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if sorted_nums[mid] == target:
            return mid
        elif sorted_nums[mid] < target:
            lo = mid + 1          # throw away the left half
        else:
            hi = mid - 1          # throw away the right half
    return -1

# O(n): look at every element once
def find_max(nums):
    best = nums[0]
    for x in nums:
        if x > best:
            best = x
    return best

# O(n log n): Python's built-in sort
def sort_it(nums):
    return sorted(nums)

# O(n²): every element paired with every later element
def all_pairs(nums):
    pairs = []
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            pairs.append((nums[i], nums[j]))
    return pairs

banner(f"One example of each class, nums = {nums}")
show_table([
    ["get_first(nums)",             "O(1)",       get_first(nums)],
    ["binary_search(sorted(...), 7)", "O(log n)", f"index {binary_search(sorted(nums), 7)} in {sorted(nums)}"],
    ["find_max(nums)",              "O(n)",       find_max(nums)],
    ["sort_it(nums)",               "O(n log n)", sort_it(nums)],
    ["all_pairs(nums)",             "O(n²)",      f"{len(all_pairs(nums))} pairs (7·6/2 = 21)"],
], ["call", "complexity", "result"])

### How many steps does binary search really take?

Each step halves the remaining list, so the number of steps is about log₂(n). Even for a billion items, that's only about 30 steps.

In [ ]:
rows = []
for n in [8, 1_000, 1_000_000, 1_000_000_000]:
    steps, remaining = 0, n
    while remaining > 1:
        remaining //= 2       # each binary-search step halves what's left
        steps += 1
    rows.append([f"{n:,}", steps])

banner("Binary search: how many halvings until 1 item is left?")
show_table(rows, ["n", "steps ≈ log₂(n)"])

---
## 4. Time vs. space: the core trade-off

Here is the same problem (**Two Sum**: find two numbers that add up to a target) solved in two ways.

| | Brute force | HashMap |
|---|---|---|
| Idea | Checks every pair | Remembers what it has seen |
| Time | **O(n²)** | **O(n)** |
| Extra space | **O(1)** | **O(n)** |

> 💡 **Trade space for speed:** use memory to avoid repeated work.

Don't worry about the details of `two_sum_hashmap` yet. Section 10 walks through it line by line.

In [ ]:
def two_sum_brute(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):    # try every pair (i, j)
            if nums[i] + nums[j] == target:
                return [i, j]
    return None

def two_sum_hashmap(nums, target):
    seen = {}                                # value -> index
    for i, x in enumerate(nums):
        if target - x in seen:               # is the partner already stored?
            return [seen[target - x], i]
        seen[x] = i
    return None

banner("Same answer, different cost: nums = [2, 7, 11, 15], target = 9")
show_table([
    ["two_sum_brute",   "O(n²)", "O(1)", two_sum_brute([2, 7, 11, 15], 9)],
    ["two_sum_hashmap", "O(n)",  "O(n)", two_sum_hashmap([2, 7, 11, 15], 9)],
], ["function", "time", "extra space", "result"])

### Race them

We'll build a worst-case input: the only matching pair is the **last two** elements, so the brute force has to try almost every pair. Timings will differ on your machine — watch the **ratio**, not the exact numbers.

✏️ **Try it:** change the sizes. (Careful going above ~5,000 for brute force — it gets slow fast!)

In [ ]:
def worst_case(n):
    nums = list(range(1, n - 1)) + [10**9, 10**9 + 1]   # only the last two sum to the target
    return nums, 2 * 10**9 + 1

def time_it(fn, *args):
    start = time.perf_counter()
    fn(*args)
    return time.perf_counter() - start

rows = []
for n in [500, 1_000, 2_000, 4_000]:          # ✏️ Try it: change these sizes
    nums, target = worst_case(n)
    t_brute = time_it(two_sum_brute, nums, target)
    t_hash = time_it(two_sum_hashmap, nums, target)
    rows.append([f"{n:,}", f"{t_brute:.4f}", f"{t_hash:.6f}", f"{t_brute / t_hash:,.0f}×"])

banner("Race: brute force vs. hashmap (worst-case input)")
show_table(rows, ["n", "brute force (s)", "hashmap (s)", "speed-up"])

Each time n doubles, the brute-force time roughly **quadruples** (O(n²)) while the hashmap time roughly **doubles** (O(n)). That's the doubling test from Section 2, live.

---
## 5. 🎮 Quiz: what's the complexity?

*(We play this live during the session. This is a copy you can keep and redo later.)*

For each snippet, work out the **time complexity** in terms of n = `len(nums)`. Write your answer down, then click **Reveal answer**.

**Q1.**
```python
def q1(nums):
    total = 0
    for x in nums:
        total += x
    return total
```
<details><summary>Reveal answer</summary>

**O(n)** — one pass through the list.

</details>

**Q2.**
```python
def q2(nums):
    return nums[len(nums) // 2]
```
<details><summary>Reveal answer</summary>

**O(1)** — indexing into a list is a single step, however long the list is.

</details>

**Q3.**
```python
def q3(nums):
    for x in nums:
        for y in nums:
            print(x, y)
```
<details><summary>Reveal answer</summary>

**O(n²)** — for each of the n elements, the inner loop runs n times.

</details>

**Q4.**
```python
def q4(nums):
    for x in nums:
        print(x)
    for y in nums:
        print(y)
```
<details><summary>Reveal answer</summary>

**O(n)** — two loops *one after the other* is n + n = 2n, and we drop constants. Loops **side by side add**; loops **inside each other multiply**.

</details>

**Q5.**
```python
def q5(n):
    i = 1
    while i < n:
        i *= 2
```
<details><summary>Reveal answer</summary>

**O(log n)** — `i` doubles every step, so it reaches n after about log₂(n) steps.

</details>

**Q6.**
```python
def q6(nums, target):
    return target in nums          # nums is a list
```
<details><summary>Reveal answer</summary>

**O(n)** — `in` on a **list** scans element by element. (On a **set** it would be O(1) on average — see Section 6.)

</details>

**Q7.**
```python
def q7(nums):
    nums.sort()
    for x in nums:
        print(x)
```
<details><summary>Reveal answer</summary>

**O(n log n)** — sorting is O(n log n), the loop is O(n). Add them and keep the biggest term: O(n log n + n) = **O(n log n)**.

</details>

**Q8.**
```python
def q8(nums):
    for i in range(len(nums)):
        for j in range(5):
            print(nums[i] + j)
```
<details><summary>Reveal answer</summary>

**O(n)** — the inner loop always runs 5 times, a constant. 5n → O(n). Nested loops only mean O(n²) when *both* depend on n.

</details>

**Q9.** *(Spot the hidden loop)*
```python
def q9(nums):
    result = []
    for x in nums:
        if x not in result:     # result is a LIST
            result.append(x)
    return result
```
<details><summary>Reveal answer</summary>

**O(n²)**. `x not in result` scans the list, so there is a loop hidden inside the loop. Change `result` to a **set** and the check becomes O(1) on average, which makes the whole function **O(n)**.

</details>

**Q10.** *(Time **and** space)*
```python
def q10(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return True
        seen.add(x)
    return False
```
<details><summary>Reveal answer</summary>

**Time O(n), space O(n).** The loop runs at most n times and each set operation is O(1) on average. In the worst case (all values unique) the set stores all n values. You'll meet this function again in Section 8, where it solves *Contains Duplicate*.

</details>

### Rules of thumb
- **Drop constants:** O(2n) → O(n), O(5) → O(1)
- **Keep the biggest term:** O(n² + n) → O(n²)
- **Sequential steps add**, **nested steps multiply**
- **Halving each step** → O(log n)
- **Watch for hidden loops:** `x in list`, `list.index()`, `sorted()`, slicing `nums[a:b]` all do work proportional to the data

---
## 6. HashMaps & Sets: the O(1) trick

### What are they?
They store information so we can **check whether a value exists without repeatedly scanning the entire input**.

| | HashMap | Set |
|---|---|---|
| Stores | **key → value** pairs | **unique values only** |
| Python type | `dict` | `set` |
| Example | `{🍎: 3, ⭐: 1, 🔑: 7}` | `{2, 5, 7, 9}`, and adding another `5` is rejected ❌ |
| Lookup | `d[key]`, `key in d` → O(1) average | `x in s` → O(1) average |

Under the hood, both use a *hash function* to jump almost straight to where a value is stored. That is why a lookup takes **O(1) time on average** instead of a scan through everything.

### When should I think about them?
- I need to **remember values already seen** → set (or dict)
- I need to **count occurrences** → dict (frequency counter)
- I need to **retrieve information using a key** → dict
- I need **fast membership checks** → set

In [ ]:
# ---- HashMap (dict): key -> value ----
inventory = {"apple": 3, "star": 1, "key": 7}       # key -> value: 🍎 3, ⭐ 1, 🔑 7
steps = []

steps.append(['inventory["apple"]', "look up by key", inventory["apple"]])
inventory["banana"] = 2
steps.append(['inventory["banana"] = 2', "insert a new key", short(inventory, 50)])
inventory["apple"] += 1
steps.append(['inventory["apple"] += 1', "update a value", short(inventory, 50)])
steps.append(['"star" in inventory', "membership (checks keys)", "star" in inventory])
steps.append(['inventory.get("pear", 0)', "safe lookup with default", inventory.get("pear", 0)])

banner("HashMap (dict): key → value")
show_table(steps, ["code", "what it does", "result"])

In [ ]:
# ---- Set: unique values only ----
s = {2, 5, 7, 9}                                    # unique values only
steps = []

s.add(5)
steps.append(["s.add(5)", "already there, so nothing changes", short(s)])   # sets have no order, so Python may print them in any order
s.add(4)
steps.append(["s.add(4)", "new value is added", short(s)])
steps.append(["5 in s", "membership check", 5 in s])
steps.append(["6 in s", "membership check", 6 in s])
steps.append(["set([1, 2, 2, 3, 3, 3])", "quick way to remove duplicates", set([1, 2, 2, 3, 3, 3])])

banner("Set: unique values only")
show_table(steps, ["code", "what it does", "result"])

### See the difference: `in` on a list vs. a set

Same question — *"is this value here?"* — on the same data. A list has to scan; a set jumps straight there.

In [ ]:
n = 1_000_000
data_list = list(range(n))
data_set  = set(data_list)
target = n - 1                       # worst case for the list: right at the end

t_list = time_it(lambda: target in data_list)
t_set  = time_it(lambda: target in data_set)

banner(f"Is {target:,} in the collection? (n = {n:,})")
show_table([
    ["list", "O(n): scans element by element", f"{t_list:.6f}"],
    ["set",  "O(1) average: jumps straight there", f"{t_set:.6f}"],
], ["structure", "how `in` works", "time (s)"])
print(f"\n→ The set is roughly {t_list / t_set:,.0f}× faster here.")

> ⚠️ Building the set itself costs O(n) time and O(n) memory. It pays off when you'll do **many** lookups — which is exactly the situation in the problems that follow.

---
## 7. Pattern: Frequency Counter

A frequency counter stores **how many times each value appears**.

Instead of searching the entire array every time we need a count, we **update one dictionary entry as we scan**.

```
A B A C A   →   { A: 3, B: 1, C: 1 }
```

**How to think about it:**
1. What value am I counting?
2. Have I seen this value before?
3. What does its new count tell me?

→ **Count each item during one pass through the input.**

**Time: O(n) | Space: O(k)**, where k is the number of *distinct* values (k ≤ n).

In [ ]:
def frequency(items):
    """Return a dict mapping each item to how many times it appears."""
    counts = {}
    for item in items:
        counts[item] = counts.get(item, 0) + 1    # 0 if unseen, then +1
    return counts

# Trace: watch ONE dictionary entry change per step
counts, rows = {}, []
for step, item in enumerate("ABACA", 1):
    seen_before = item in counts
    counts[item] = counts.get(item, 0) + 1
    rows.append([step, item, "yes" if seen_before else "no", counts[item], short(dict(counts))])

banner('Frequency counter trace: "ABACA"')
show_table(rows, ["step", "item", "seen before?", "new count", "counts so far"])

print()
banner("frequency() on other inputs")
show_table([
    ['"ABACA"', frequency("ABACA")],
    ["[3, 1, 3, 3, 2]", frequency([3, 1, 3, 3, 2])],
    ['"mississippi"', frequency("mississippi")],
], ["input", "counts"])

Python also has a built-in that does exactly this:

In [ ]:
from collections import Counter

c = Counter("ABACA")
banner("collections.Counter: a ready-made frequency counter")
show_table([
    ['Counter("ABACA")',       c],
    ["c.most_common(1)",       c.most_common(1)],
    ['c["A"]',                 c["A"]],
    ['c["Z"]  (missing key)',  f'{c["Z"]}  (no KeyError, just 0)'],
], ["code", "result"])

### Worked example: Valid Anagram

Two strings are **anagrams** if they use exactly the same letters, the same number of times (`"listen"` and `"silent"`). With a frequency counter, the check is a single comparison of two dictionaries.

In [ ]:
def is_anagram(s, t):
    if len(s) != len(t):          # different lengths can't be anagrams
        return False
    return frequency(s) == frequency(t)

for s, t in [("listen", "silent"), ("rat", "car")]:
    banner(f"is_anagram({s!r}, {t!r}) → {is_anagram(s, t)}")
    # sort each dict's letters so the two rows are easy to compare
    show_table([[w, dict(sorted(frequency(w).items()))] for w in (s, t)], ["word", "letter counts"])
    print()

*Alternative:* `sorted(s) == sorted(t)` also works, but sorting costs **O(n log n)**. The counter is **O(n)**.

---
## 8. Problem: Contains Duplicate

### What do we solve?
Given an integer array, return `True` if any value appears **more than once**. Otherwise return `False`.

```
[1, 2, 3, 1]  →  True   (1 appears twice)
[1, 2, 3, 4]  →  False
```

### Algorithm
Start with an empty set called `seen`.
1. If the current number is **already in `seen`**, we have found a duplicate → return `True`.
2. Otherwise, add it to `seen`.
3. If the loop finishes, every value was unique → return `False`.

> 💡 When the question asks whether something **has appeared before**, think: *"Can I store the values I have already seen?"*

### Code

In [ ]:
def contains_duplicate(nums):
    seen = set()

    for num in nums:
        if num in seen:
            return True

        seen.add(num)

    return False

### Trace it

Watch `seen` grow step by step:

In [ ]:
def contains_duplicate_traced(nums):
    """Same algorithm as above, but prints a row for every step."""
    seen, rows, result = set(), [], False
    for num in nums:
        already = num in seen
        rows.append([num, short(seen) if seen else "{}", "YES → duplicate!" if already else "no → add it"])
        if already:
            result = True
            break
        seen.add(num)
    banner(f"Trace: contains_duplicate({nums})")
    show_table(rows, ["num", "seen (before)", "already in seen?"])
    print(f"Result: {result}\n")
    return result

contains_duplicate_traced([1, 2, 3, 1])
contains_duplicate_traced([4, 5, 6]);

**Time: O(n), Space: O(n)** — one pass, and in the worst case (all unique) the set holds every value.

### Comparing approaches

| Approach | Time | Space | Notes |
|---|---|---|---|
| Check every pair | O(n²) | O(1) | Too slow for large n |
| Sort, then compare neighbours | O(n log n) | O(n) | Duplicates end up next to each other |
| **Set of seen values** | **O(n)** | **O(n)** | Stops early as soon as it finds one |
| `len(set(nums)) != len(nums)` | O(n) | O(n) | Neat one-liner, but always processes the whole list |

In [ ]:
def contains_duplicate_sort(nums):
    nums = sorted(nums)                  # duplicates are now neighbours
    for i in range(1, len(nums)):
        if nums[i] == nums[i - 1]:
            return True
    return False

def contains_duplicate_oneliner(nums):
    return len(set(nums)) != len(nums)   # a set drops duplicates, so the sizes differ

banner("Three ways, same answers")
show_table(
    [[nums, contains_duplicate(nums), contains_duplicate_sort(nums), contains_duplicate_oneliner(nums)]
     for nums in ([1, 2, 3, 1], [1, 2, 3, 4])],
    ["nums", "set of seen", "sort first", "one-liner"],
)

---
## 9. Pattern: Complement Lookup

A **complement** is the value **needed to complete a condition**.

For Two Sum:

$$\text{complement} = \text{target} - \text{current number}$$

If the target is **9** and the current number is **7**, we need a **2**.

If we store the values we've already seen, we can check for that 2 **immediately** instead of trying every possible pair.

**How to think about it:**
1. What value would complete the target?
2. Have I already seen that value?
3. What information should I store with it? *(For Two Sum: its **index**, because that's what we have to return.)*

→ **Search for the missing value instead of trying every possible pair.**

A hash-map lookup takes **O(1) time on average**, so each "have I seen it?" question is cheap.

In [ ]:
target = 9
banner(f"What does each number need? (target = {target})")
show_table(
    [[current, f"{target} − {current}", target - current] for current in [2, 7, 11, 15]],
    ["current", "complement", "needed partner"],
)

---
## 10. Problem: Two Sum

### What do we solve?
Given an array and a target, return the **indices** of two numbers whose **sum equals the target**.

```
nums = [2, 7, 11, 15], target = 9
index:  0  1
2 + 7 = 9   →   return [0, 1]
```

### Algorithm
For each number:
1. Calculate the complement: `target - number`
2. Check whether the complement is already in the dictionary
3. **If found**, return both indices
4. Otherwise, store the current number and its index

The dictionary stores **`number: index`**. This lets us find the required partner without checking every pair.

### Code

In [ ]:
def two_sum(nums, target):
    seen = {}                          # number -> index where we saw it

    for i, num in enumerate(nums):
        needed = target - num          # 1. the complement

        if needed in seen:             # 2. O(1) average lookup
            return [seen[needed], i]   # 3. found: return both indices

        seen[num] = i                  # 4. remember this number for later

    return None                        # no pair found

### Trace it

In [ ]:
def two_sum_traced(nums, target):
    """Same algorithm as two_sum, but prints a row for every step."""
    seen, rows, result = {}, [], None
    for i, num in enumerate(nums):
        needed = target - num
        found = needed in seen
        action = f"✔ found {needed} at index {seen[needed]}" if found else f"✘ store {num}: {i}"
        rows.append([i, num, needed, short(dict(seen)), action])
        if found:
            result = [seen[needed], i]
            break
        seen[num] = i
    banner(f"Trace: two_sum({nums}, target={target})")
    show_table(rows, ["i", "num", "needed", "seen (before)", "action"])
    print(f"Result: {result}\n")
    return result

two_sum_traced([2, 7, 11, 15], 9)
two_sum_traced([3, 2, 4], 6);

### Why check **before** storing?

Look at the order of steps 2 and 4. We check the dictionary **first** and store the current number afterwards. This stops a number from being paired with **itself**.

Example: `nums = [3, 2, 4]`, `target = 6`. If we stored `3` first and then checked, we would find `6 - 3 = 3` in the dictionary and wrongly return `[0, 0]`.

Genuine repeats still work: `[3, 3]` with target 6 returns `[0, 1]`, because the first 3 is already stored when we reach the second one.

**Time: O(n), Space: O(n).** Brute force takes **O(n²)** time.

| Approach | Time | Space |
|---|---|---|
| Brute force (every pair) | O(n²) | O(1) |
| **HashMap complement lookup** | **O(n)** | **O(n)** |

---
## 11. Practice problems ✏️

Each exercise has four parts:

1. **Problem**: what to solve, with examples.
2. **Your code**: replace `pass` with your solution and run the cell.
3. **Check cell**: run it to compare your answers with the correct ones (✅ / ❌).
4. **Reveal answer key**: a fully explained solution. Give the problem a real try first!

Use tonight's patterns and aim for **O(n)** time.

| # | Problem | Pattern | Difficulty |
|---|---|---|---|
| 1 | First Unique Character | Frequency Counter | 🟢 Easy |
| 2 | Intersection of Two Arrays | Set membership | 🟢 Easy |
| 3 | Majority Element | Frequency Counter | 🟢 Easy |
| 4 | Pair With Difference K | Complement Lookup | 🟡 Medium |
| 5 | Group Anagrams | HashMap of lists | 🟡 Medium |

### Exercise 1 — First Unique Character 🟢 *(Frequency Counter)*

Given a string `s`, return the **index** of the first character that appears **exactly once**. If there is no such character, return `-1`.


| Input | Output | Why |
|---|---|---|
| `"leetcode"` | `0` | `'l'` appears once |
| `"loveleetcode"` | `2` | `'l'` and `'o'` repeat, `'v'` doesn't |
| `"aabb"` | `-1` | every character repeats |

*Hint: use two passes. The first counts every character; the second finds the first character with a count of 1.*

In [ ]:
def first_unique_char(s):
    """
    Return the index of the first character in s that appears exactly once,
    or -1 if there is none.

    Example: first_unique_char("leetcode")  →  0
    Pattern: Frequency Counter. Target: O(n) time.
    """
    pass  # ✏️ replace with your code

In [ ]:
check(first_unique_char, [
    (("leetcode",), 0),
    (("loveleetcode",), 2),
    (("aabb",), -1),
    (("z",), 0),
    (("",), -1),
])

<details><summary>Reveal answer key</summary>

```python
def first_unique_char(s):
    # Pass 1: frequency counter. counts[ch] = how many times ch appears in s.
    counts = {}
    for ch in s:
        counts[ch] = counts.get(ch, 0) + 1

    # Pass 2: walk the string IN ORDER. The first character with a count
    # of 1 is the answer. (Walking the dict would lose the string's order.)
    for i, ch in enumerate(s):
        if counts[ch] == 1:
            return i

    # Every character repeats (or s is empty).
    return -1
```
**Time O(n)**: two passes is 2n steps, which is still O(n). **Space O(k)**: one entry per different character.

</details>

### Exercise 2 — Intersection of Two Arrays 🟢 *(Set membership)*

Return a **sorted** list of the **unique** values that appear in **both** `a` and `b`. (Below, n = `len(a)` and m = `len(b)`.)


| `a` | `b` | Output |
|---|---|---|
| `[1, 2, 2, 1]` | `[2, 2]` | `[2]` |
| `[4, 9, 5]` | `[9, 4, 9, 8, 4]` | `[4, 9]` |
| `[1, 2, 3]` | `[4, 5, 6]` | `[]` |

*Hint: how much does `x in b` cost if `b` is a list? And if it's a set?*

In [ ]:
def intersection(a, b):
    """
    Return a sorted list of the unique values found in both a and b.

    Example: intersection([4, 9, 5], [9, 4, 9, 8, 4])  →  [4, 9]
    Pattern: Set membership. Target: O(n + m) time (plus sorting the output).
    """
    pass  # ✏️ replace with your code

In [ ]:
check(intersection, [
    (([1, 2, 2, 1], [2, 2]), [2]),
    (([4, 9, 5], [9, 4, 9, 8, 4]), [4, 9]),
    (([1, 2, 3], [4, 5, 6]), []),
    (([], [1]), []),
])

<details><summary>Reveal answer key</summary>

```python
def intersection(a, b):
    # Turn b into a set ONCE (O(m)), so each "x in b_set" check is O(1).
    b_set = set(b)

    # Collect matches in a set as well, so duplicates in a are only kept once.
    result = set()
    for x in a:                 # O(n) loop...
        if x in b_set:          # ...with an O(1) check inside
            result.add(x)

    return sorted(result)       # the problem asks for sorted output

# Shorter version using set intersection:
#     return sorted(set(a) & set(b))
```
**Time O(n + m)** to build and check the sets, plus O(k log k) to sort the k matches. **Space O(n + m)**.
If you wrote `x in b` with `b` still a **list**, the loop has a hidden O(m) scan inside it, so the total is **O(n·m)**.

</details>

### Exercise 3 — Majority Element 🟢 *(Frequency Counter)*

Return the value that appears **more than n / 2 times**, where n = `len(nums)`. You may assume that such a value always exists.


| Input | Output |
|---|---|
| `[3, 2, 3]` | `3` |
| `[2, 2, 1, 1, 1, 2, 2]` | `2` |

In [ ]:
def majority_element(nums):
    """
    Return the value that appears more than len(nums) // 2 times.

    Example: majority_element([2, 2, 1, 1, 1, 2, 2])  →  2
    Pattern: Frequency Counter. Target: O(n) time.
    """
    pass  # ✏️ replace with your code

In [ ]:
check(majority_element, [
    (([3, 2, 3],), 3),
    (([2, 2, 1, 1, 1, 2, 2],), 2),
    (([5],), 5),
    (([1, 1, 2],), 1),
])

<details><summary>Reveal answer key</summary>

```python
def majority_element(nums):
    half = len(nums) // 2
    counts = {}
    for x in nums:
        # Frequency counter: bump x's count by one.
        counts[x] = counts.get(x, 0) + 1
        # "What does its new count tell me?" If x has passed the halfway
        # mark, it must be the majority, so we can stop early.
        if counts[x] > half:
            return x
```
**Time O(n), Space O(k).**
*Bonus:* look up the **Boyer–Moore voting algorithm**, which solves this in O(1) extra space.

</details>

### Exercise 4 — Pair With Difference K 🟡 *(Complement Lookup)*

Return `True` if there are two **different positions** `i ≠ j` with `nums[i] - nums[j] == k`. Otherwise return `False`. Assume `k ≥ 0`.


| `nums` | `k` | Output | Why |
|---|---|---|---|
| `[1, 5, 3, 4, 2]` | `3` | `True` | 5 − 2 = 3 |
| `[1, 2, 3]` | `5` | `False` | no pair is 5 apart |
| `[7]` | `0` | `False` | an element can't pair with itself |
| `[7, 7]` | `0` | `True` | two different positions |

*Hint: this works like Two Sum. For each `x`, which earlier values would complete the condition? There are **two** candidates.*

In [ ]:
def has_pair_with_difference(nums, k):
    """
    Return True if nums[i] - nums[j] == k for some positions i != j.

    Example: has_pair_with_difference([1, 5, 3, 4, 2], 3)  →  True
    Pattern: Complement Lookup. Target: O(n) time.
    """
    pass  # ✏️ replace with your code

In [ ]:
check(has_pair_with_difference, [
    (([1, 5, 3, 4, 2], 3), True),
    (([1, 2, 3], 5), False),
    (([7], 0), False),
    (([7, 7], 0), True),
    (([10, 1], 9), True),
])

<details><summary>Reveal answer key</summary>

```python
def has_pair_with_difference(nums, k):
    seen = set()                      # values at EARLIER positions
    for x in nums:
        # x is either the larger number of the pair (its partner is x - k)
        # or the smaller one (its partner is x + k). Check both complements.
        if (x - k) in seen or (x + k) in seen:
            return True
        # Store x only AFTER checking, so x can never pair with itself.
        # This is why [7] with k = 0 correctly returns False.
        seen.add(x)
    return False
```
It has the same shape as Two Sum: **check first, then store**. **Time O(n), Space O(n).**

</details>

### Exercise 5 — Group Anagrams 🟡 *(HashMap: key → list)*

Group the words that are anagrams of each other. Return a list of groups. To make the answer easy to check, **sort the words inside each group, then sort the list of groups**.


| Input | Output |
|---|---|
| `["eat", "tea", "tan", "ate", "nat", "bat"]` | `[["ate", "eat", "tea"], ["bat"], ["nat", "tan"]]` |
| `[""]` | `[[""]]` |

*Hint: anagrams become the **same string** when you sort their letters. What could you use as a dictionary key?*

In [ ]:
def group_anagrams(words):
    """
    Group anagrams together. Sort the words inside each group, then sort the groups.

    Example: group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])  →  [['ate', 'eat', 'tea'], ['bat'], ['nat', 'tan']]
    Pattern: HashMap where the key is a "signature" and the value is a list.
    """
    pass  # ✏️ replace with your code

In [ ]:
check(group_anagrams, [
    ((["eat", "tea", "tan", "ate", "nat", "bat"],), [["ate", "eat", "tea"], ["bat"], ["nat", "tan"]]),
    (([""],), [[""]]),
    ((["a"],), [["a"]]),
    (([],), []),
])

<details><summary>Reveal answer key</summary>

```python
def group_anagrams(words):
    groups = {}                                  # signature -> list of words
    for word in words:
        # All anagrams share the same sorted letters: "eat", "tea", "ate" -> "aet".
        signature = "".join(sorted(word))
        # Retrieve information using a key: add the word to its group's list.
        groups.setdefault(signature, []).append(word)

    # Sort the output so it matches the expected answer exactly.
    return sorted(sorted(group) for group in groups.values())
```
**Time O(n · m log m)**, where n is the number of words and m is the length of the longest word (we sort each word's letters). **Space O(n · m)** for the dictionary.
*Faster key:* a tuple of 26 letter counts, built with a frequency counter, gives O(n · m).

</details>

---
## Recap

| Concept | Remember |
|---|---|
| **Big-O** | Growth rate, not speed. Ask what happens when n doubles. |
| **Complexity ladder** | O(1) < O(log n) < O(n) < O(n log n) < O(n²) |
| **Rules of thumb** | Drop constants · keep the biggest term · sequential steps add, nested steps multiply · watch for hidden loops |
| **Time vs space** | Spend memory to avoid repeated work. |
| **HashMap / Set** | O(1) average lookups: remember, count, retrieve by key, check membership. |
| **Frequency counter** | Count each item in one pass. Time O(n), space O(k). |
| **Seen-set** | "Has this appeared before?" → store what you've seen. |
| **Complement lookup** | Search for the missing value instead of trying every pair. Check first, then store. |

### Keep practising
These LeetCode problems use tonight's patterns (search for the titles):
- *Contains Duplicate* · *Two Sum* · *Valid Anagram* · *First Unique Character in a String*
- *Intersection of Two Arrays* · *Majority Element* · *Ransom Note* · *Group Anagrams*

➡️ **Next:** open **`Night1_Part2_Patterns.ipynb`** for two pointers, sliding windows and the interview method.